<a href="https://colab.research.google.com/github/pyknjc/colab-test/blob/test/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lấy dữ liệu

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

source_folder = '/content/drive/MyDrive/document_image'

if os.path.exists(source_folder):
    print("Đã tìm thấy thư mục!")
else:
    print("Chưa tìm thấy thư mục, hãy kiểm tra lại tên folder trên Drive nhé!")

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ĐƯỜNG DẪN MỚI TRÊN COLAB
source_folder = '/content/drive/MyDrive/document_image'
output_folder = '/content/drive/MyDrive/dataset_processed'

labels = ['0', '90', '180', '270']
for label in labels:
    os.makedirs(os.path.join(output_folder, 'train', label), exist_ok=True)

# Duyệt qua tất cả các thư mục con và file
for root, dirs, files in os.walk(source_folder):
    for img_name in tqdm(files, desc=f"Processing {os.path.basename(root)}"):
        # Kiểm tra định dạng ảnh
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        img_path = os.path.join(root, img_name)
        img = cv2.imread(img_path)

        if img is None:
            print(f"Không thể đọc file: {img_path}")
            continue

        # Tên file mới để tránh trùng lặp nếu lấy từ nhiều thư mục con
        base_name = os.path.splitext(img_name)[0]
        folder_name = os.path.basename(root)
        unique_name = f"{folder_name}_{base_name}.jpg"

        # Lưu vào 4 folder nhãn tương ứng
        cv2.imwrite(os.path.join(output_folder, 'train', '0', unique_name), img)
        cv2.imwrite(os.path.join(output_folder, 'train', '90', unique_name), cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE))
        cv2.imwrite(os.path.join(output_folder, 'train', '180', unique_name), cv2.rotate(img, cv2.ROTATE_180))
        cv2.imwrite(os.path.join(output_folder, 'train', '270', unique_name), cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE))

print("\nTạo dữ liệu xong! Kiểm tra trong Drive của bạn tại:", output_folder)

Code Training

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=5,
    zoom_range=0.1
)

train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/dataset_processed/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/dataset_processed/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

checkpoint = callbacks.ModelCheckpoint('/content/drive/MyDrive/best_model.h5', save_best_only=True)

model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=[checkpoint]
)

In [ ]:
import os

folder_path = '/content/drive/MyDrive'

if os.path.exists(folder_path):
    print(f"Các file trong thư mục {folder_path} là:")
    print(os.listdir(folder_path))
else:
    print("Lỗi: Thư mục /content/drive/MyDrive không tồn tại. Bạn đã Mount Drive chưa?")

Các file trong thư mục /content/drive/MyDrive là:
['Classroom', 'Colab Notebooks', 'document_image', 'dataset_processed', 'best_model.h5', 'test_image.jpg', '.ipynb_checkpoints']


In [ ]:
import cv2
import numpy as np
import shutil
import os
from tensorflow.keras.models import load_model

# 1. Load model (đường dẫn của bạn đã đúng)
model = load_model('/content/drive/MyDrive/best_model.h5')

def detect_and_rotate(image_path):
    # Đường dẫn tạm để copy ảnh về (local trong Colab)
    temp_path = '/content/temp_image.jpg'

    # Copy file từ Drive về bộ nhớ local của Colab
    try:
        shutil.copy(image_path, temp_path)
    except Exception as e:
        print(f"Không thể copy file: {e}")
        return None

    # Bây giờ đọc từ đường dẫn local (rất nhanh và không bị lỗi)
    img = cv2.imread(temp_path)

    if img is None:
        print("Lỗi: Không thể đọc được file ảnh.")
        return None

    # Xử lý ảnh để dự đoán
    img_resized = cv2.resize(img, (224, 224))
    img_input = np.expand_dims(img_resized / 255.0, axis=0)

    # Dự đoán
    prediction = model.predict(img_input)
    angle_idx = np.argmax(prediction)
    angles = [0, 90, 180, 270]
    angle = angles[angle_idx]

    print(f"Ảnh được dự đoán là xoay: {angle} độ")

    # Xoay ảnh dựa trên kết quả
    if angle == 90: result = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif angle == 180: result = cv2.rotate(img, cv2.ROTATE_180)
    elif angle == 270: result = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    else: result = img

    # Xóa file tạm sau khi dùng xong (để tiết kiệm dung lượng)
    if os.path.exists(temp_path):
        os.remove(temp_path)

    return result

# 2. Gọi hàm
# Đảm bảo đường dẫn này khớp với file bạn nhìn thấy trong sidebar bên trái
image_path = '/content/drive/MyDrive/test_image.jpg/image-5bd08790e1864.png'
result = detect_and_rotate(image_path)

# 3. Lưu kết quả
if result is not None:
    cv2.imwrite('/content/drive/MyDrive/corrected_image.jpg', result)
    print("Đã lưu ảnh đã xoay vào Drive thành công!")